# Outsourced learning without differential privacy

This notebook runs the outsourced learning (FL) baseline for the SoK experiments on accuracy, i.e., without differential privacy (DP)
We simulate MPC protocols using the [CrypTen](https://crypten.readthedocs.io/en/latest/) library and provide the code for one party training simulation and 2 parties with multiprocessing.

In [ ]:
import sys
import torch
import numpy as np
import copy
import crypten
import crypten.mpc as mpc
import crypten.communicator as comm
from crypten.config import cfg



import nest_asyncio
nest_asyncio.apply()

import os

device = "cuda" if torch.cuda.is_available() else "cpu"
# If on Apple Silicon, use MPS
# device = "mps"
# os.environ['PFL_PYTORCH_DEVICE'] = device

sys.path.append('./dataset/')
from dataset.fashion_mnist.load_preprocess import load_and_preprocess_fashion_mnist
from dataset.mnist.load_preprocess import load_and_preprocess_mnist

sys.path.append('./utils/')
from utils.models import CryptenThreeLayerNN
from utils.mpc_dpsgd_trainer import DP_TrainerEncrypted

## Model
The model used for the experiments is a 3 layers neural network. Here, the model is adapted to use CrypTen layers instead of PyTorch ones (for more details see the code in `utils/models.py`).

In [ ]:
model = CryptenThreeLayerNN(
    input_size=784,
	hidden_size=100,
	output_size=10
)

# The model is copied in model_2p since this model will be used for the two parties experiments with multiprocessing.
model_2p = copy.deepcopy(model)
model_enc = model.encrypt()

## Dataset
The experiments can be run on either MNIST or Fashion-MNIST datasets by changing the `dataset` variable below.

### Pre-processing
For each dataset, we perform standard pre-processing, i.e., scaling and normalization. More details in `dataset/mnist/load_preprocess_mnist.py` and `dataset/fashion_mnist/load_preprocess_fashion_mnist.py`.

In [ ]:
dataset_name = 'mnist' 
#dataset_name = 'fashion_mnist' 

if dataset_name == 'mnist':
    	train_data, val_data = load_and_preprocess_mnist(
	    scaling=True,
	    normalization=True
	)
elif dataset_name == 'fashion_mnist':
	train_data, val_data = load_and_preprocess_fashion_mnist(
		scaling=True,
		normalization=True
	)
else:
	raise ValueError(f"Unsupported dataset name: {dataset_name}")

x_train, y_train = train_data.data, train_data.targets
x_val, y_val = val_data.data, val_data.targets

y_train = y_train.squeeze()
y_val = y_val.squeeze()

## Training

### Hyperparameters

In [ ]:
batch_size = 32
learning_rate = 0.01
epochs = 30

### One party simulation
The one party simulation can be used to test and validate the training code.

In [ ]:
torch.random.manual_seed(0)
np.random.seed(0)

crypten.init()

# CrypTen requires labels to be one-hot encoded
y_train_onehot = torch.nn.functional.one_hot(y_train, num_classes=10)
y_val_onehot = torch.nn.functional.one_hot(y_val, num_classes=10)

x_train_enc = crypten.cryptensor(x_train)
y_train_enc_onehot = crypten.cryptensor(y_train_onehot)

# For validation, the trainer requires both the version of the labels
# The one-hot encoded labels to compute the loss using CrypTen functions
# The integer labels to compute utility scores
x_val_enc = crypten.cryptensor(x_val)
y_val_enc_onehot = crypten.cryptensor(y_val_onehot)
y_val_enc = crypten.cryptensor(y_val)


# We leverage our custom DP trainer to perform secure training with CrypTen 
non_dp_trainer = DP_TrainerEncrypted(
    model=model_enc,
    optimizer_type='sgd', # The optimizer type defines DP or non-DP training
    batch_size=batch_size,
    lr = learning_rate,
	num_epochs=epochs,
    verbose = False,
	device=device,
    num_labels=10,
)

exp_iterations = 12 # Standard value is 8. Increased to avoid numerical errors.
with cfg.temp_override({"functions.exp_iterations": exp_iterations}):
	non_dp_trainer.train_and_validate(
		x = x_train_enc, 
		y = y_train_enc_onehot,
		x_val = x_val_enc,
		y_val = y_val_enc,
		y_val_onehot = y_val_enc_onehot,
		validation_freq=1
	)

### Two parties simulation
The two parties simulation is used to evaluate the accuracy drop compared to plain-text. While for the one party simulation no numeric error can occur, the two party setting can suffer from numeric errors due to fixed precision, e.g. underflows, overflows. The two parties are simulated with two different processes.

In [ ]:
@mpc.run_multiprocess(world_size=2)
def train_2p():
	rank = comm.get().get_rank()

	model_enc_2p = model_2p.encrypt()
	non_dp_trainer_2p = DP_TrainerEncrypted(
		model=model_enc_2p,
		optimizer_type='sgd', # The optimizer type defines DP or non-DP training
		batch_size=batch_size,
		lr = learning_rate,
		num_epochs=epochs,
		verbose = False,
		device=device,
		num_labels=10,
	)

	
	x_train_enc = crypten.cryptensor(x_train)
	crypten.print("Process: ",rank, "X encrypted")
	y_train_enc_onehot = crypten.cryptensor(y_train_onehot)
	crypten.print("Process: ",rank, "Y encrypted")

	x_val_enc = crypten.cryptensor(x_val)
	crypten.print("Process: ",rank, "X val encrypted")
	y_val_enc_onehot = crypten.cryptensor(y_val_onehot)
	crypten.print("Process: ",rank, "Y val one-hot encrypted")
	y_val_enc = crypten.cryptensor(y_val)
	crypten.print("Process: ",rank, "Y val encrypted")

	crypten.print("Process: ",rank, "Shapes: ",x_train.shape, y_train.shape)

	exp_iterations = 12 # Standard value is 8 
	with cfg.temp_override({"functions.exp_iterations": exp_iterations}):
		non_dp_trainer_2p.train_and_validate(
			x = x_train_enc, 
			y = y_train_enc_onehot,
			x_val = x_val_enc,
			y_val = y_val_enc,
			y_val_onehot = y_val_enc_onehot,
			validation_freq=1,
			batched=True
	)
		

train_2p()